In [ ]:
# CELL 1: Kaggle repo setup
print('=== CELL 1: Kaggle repo setup ===')

import os
from pathlib import Path

REPO_URL = 'https://github.com/DanielQH07/TranSTR_Causal.git'
REPO_NAME = 'TranSTR_Causal'
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO_DIR = WORKING_DIR / REPO_NAME


def has_repo_files(p):
    p = Path(p)
    return (p / 'DataLoader.py').exists() and (p / 'networks' / 'model.py').exists()

if has_repo_files(Path.cwd()):
    print(f'Using current repo: {Path.cwd()}')
else:
    if not REPO_DIR.exists():
        print(f'Cloning {REPO_URL}...')
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f'Changed directory to repo root: {Path.cwd()}')

if not has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repo files not found in {Path.cwd()}. Expected DataLoader.py and networks/model.py at repo root.')

print(f'Working directory: {Path.cwd()}')


In [ ]:
# CELL 2: Dependencies (Kaggle)
print('=== CELL 2: Dependencies ===')

import os, sys, subprocess, importlib

# Fix Kaggle base image conflict: preinstalled `transformers` requires
# huggingface-hub<1.0, but Kaggle ships huggingface-hub==1.x by default,
# causing ImportError when importing transformers.
# Pin huggingface_hub<1.0 (same as the working Colab setup) and ensure
# transformers is installed/compatible.
print('Pinning huggingface_hub<1.0 and ensuring transformers is compatible...')
get_ipython().system('pip install -q "huggingface_hub<1.0" "transformers>=4.41,<5.0" wandb')

# Force-reload in case transformers/huggingface_hub were already imported
for mod in list(sys.modules):
    if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
        del sys.modules[mod]

import huggingface_hub, transformers
print(f'huggingface_hub=={huggingface_hub.__version__}')
print(f'transformers=={transformers.__version__}')

import wandb
from huggingface_hub import HfApi, hf_hub_download, list_repo_tree

# Kaggle training/evaluation does not need HF_TOKEN by default.
# Public HuggingFace downloads work without login. Set UPLOAD_TO_HF=True only if you want to push checkpoints.
UPLOAD_TO_HF = False

wandb_key = "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"
WANDB_PROJECT = 'transtr-Origin-causalvid'
WANDB_ENTITY = None
if wandb_key:
    wandb.login(key=wandb_key)
    print('W&B login OK')
else:
    os.environ.setdefault('WANDB_MODE', 'offline')
    print('WANDB_API_KEY not set; W&B will run offline')


In [ ]:
# CELL 2.5: Runtime hotfix for transformers tokenizer API
print('=== CELL 2.5: Patch networks/model.py for tokenizer compatibility ===')

from pathlib import Path

model_py = Path('networks/model.py')
if not model_py.exists():
    raise FileNotFoundError(f'Not found: {model_py}. Please run clone/setup cell first.')

src = model_py.read_text(encoding='utf-8')
old = "tokenized_queries = self.tokenizer.batch_encode_plus(text_queries, padding='longest', return_tensors='pt')"
new = "tokenized_queries = self.tokenizer(text_queries, padding='longest', truncation=True, return_tensors='pt')"

if old in src:
    src = src.replace(old, new)
    model_py.write_text(src, encoding='utf-8')
    print('✅ Patched batch_encode_plus -> tokenizer(...)')
else:
    print('ℹ️ Patch not needed (already compatible).')

In [ ]:
# CELL 3: Imports
print('=== CELL 3: Imports ===')
import os, torch, numpy as np, pandas as pd, tarfile, shutil, json
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
print('Imports OK')

In [ ]:
# CELL 4: Train/Eval functions (No AMP - Safe for DeBERTa)
print('=== CELL 4 ===')

def train_epoch(model, optimizer, loader, xe, device, scaler=None, accumulation_steps=1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        ff, of, q, a, ans_id, _ = batch
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        # No autocast - DeBERTa has issues with fp16
        out = model(ff, of, q, a)
        loss = xe(out, tgt)
        (loss / accumulation_steps).backward()

        if step % accumulation_steps == 0 or step == len(loader):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        correct += (out.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

    return total_loss / len(loader), correct / total * 100

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _ = batch
            out = model(ff.to(device), of.to(device), q, a)
            correct += (out.argmax(-1) == ans_id.to(device)).sum().item()
            total += ans_id.size(0)
    return correct / total * 100

print('Functions defined (No AMP - DeBERTa safe, gradient accumulation supported)')


In [ ]:
# CELL 5 + 6: Setup Kaggle Paths & Config
print('=== CELL 5+6: Kaggle Paths & Config ===')

from pathlib import Path
import os, torch

# Kaggle datasets must be added to the notebook input panel.
KAGGLE_INPUT = Path('/kaggle/input')
DATASET_ROOTS = {
    'vit': KAGGLE_INPUT / 'vit-features-full-merged',
    'obj': KAGGLE_INPUT / 'object-detection-causal-full',
    'anno': KAGGLE_INPUT / 'text-annotation',
    'split': KAGGLE_INPUT / 'casual-vid-data-split',
}

# Override these manually if your Kaggle dataset names differ.
VIT_ROOT_OVERRIDE = None
OBJ_ROOT_OVERRIDE = None
ANNO_ROOT_OVERRIDE = None
SPLIT_ROOT_OVERRIDE = None


def _find_dir_containing(root, target_name):
    root = Path(root)
    if not root.exists():
        return None
    if root.name.lower() == target_name.lower():
        return str(root)
    for p in root.rglob('*'):
        if p.is_dir() and p.name.lower() == target_name.lower():
            return str(p)
    return None


def _find_dir_with_ext(root, ext):
    root = Path(root)
    if not root.exists():
        return None
    counts = {}
    for p in root.rglob(f'*{ext}'):
        parent = str(p.parent)
        counts[parent] = counts.get(parent, 0) + 1
    if not counts:
        return None
    return max(counts.items(), key=lambda x: x[1])[0]


def _resolve_dataset_root(kind, override=None):
    if override and Path(override).exists():
        return Path(override)
    default = DATASET_ROOTS[kind]
    if default.exists():
        return default
    if KAGGLE_INPUT.exists():
        slug_hint = default.name.lower()
        matches = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir() and slug_hint in p.name.lower()]
        if matches:
            return matches[0]
    raise FileNotFoundError(f'Missing Kaggle input dataset for {kind}: expected {default}. Add it in Kaggle or set the override variable.')

print('\nResolving datasets from /kaggle/input (no KaggleHub downloads)...')
vit_root = _resolve_dataset_root('vit', VIT_ROOT_OVERRIDE)
obj_root = _resolve_dataset_root('obj', OBJ_ROOT_OVERRIDE)
anno_root = _resolve_dataset_root('anno', ANNO_ROOT_OVERRIDE)
split_root = _resolve_dataset_root('split', SPLIT_ROOT_OVERRIDE)

VIT_FEATURE_PATH = _find_dir_with_ext(vit_root, '.pt') or str(vit_root)
OBJ_FEATURE_PATH = _find_dir_with_ext(obj_root, '.pkl') or str(obj_root)
ANNOTATION_PATH = _find_dir_containing(anno_root, 'QA') or str(anno_root)
SPLIT_DIR = _find_dir_containing(split_root, 'split') or str(split_root)

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('\n--- Path Verification ---')

def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:5]
        print(f'OK {name}')
        print(f'   Path: {path}')
        print(f'   Sample: {items}')
        return True
    print(f'MISSING {name}')
    print(f'   Path: {path}')
    return False

all_ok = True
all_ok &= verify_path('ViT Features', VIT_FEATURE_PATH)
all_ok &= verify_path('Object Features', OBJ_FEATURE_PATH)
all_ok &= verify_path('Annotations', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)
if not all_ok:
    raise FileNotFoundError('One or more Kaggle input paths are missing. Check attached datasets or override paths above.')

RUN_TRAINING = True
HF_REPO_ID = 'DanielQ07/transtr-causalvid-weights'
HF_MODEL_FILENAME = 'best_model.ckpt'
LAST_CKPT_FILENAME = 'last_checkpoint.ckpt'
WANDB_PROJECT = 'transtr-causalvid'

class Config:
    video_feature_root = VIT_FEATURE_PATH
    object_feature_path = OBJ_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12
    frame_feat_dim = 1024
    obj_feat_dim = 2053
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1

    bs = 8
    accumulation_steps = 4
    lr = 1e-5
    epoch = 15
    gpu = 0
    patience = 5
    gamma = 0.1
    decay = 1e-4
    n_query = 5

    hard_eval = False
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    use_amp = False
    num_workers = 4

args = Config()
set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Config loaded | bs={args.bs}, accumulation_steps={args.accumulation_steps}, effective_bs={args.bs * args.accumulation_steps}, epoch={args.epoch}')


In [ ]:
# CELL 7: Create Datasets with Verification
print('=== CELL 7: Datasets ===')

# Set to None for full training set
MAX_TRAIN_SAMPLES = None
PIN_MEMORY = torch.cuda.is_available()

print('\n--- Creating TRAIN dataset ---')
train_ds = VideoQADataset(
    split='train',
    n_query=args.n_query,
    obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt,
    topK_frame=args.topK_frame,
    max_samples=MAX_TRAIN_SAMPLES,
    verbose=True
)

print('\n--- Creating VAL dataset ---')
val_ds = VideoQADataset(
    split='val',
    n_query=args.n_query,
    obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt,
    topK_frame=args.topK_frame,
    max_samples=None,
    verbose=True
)

print('\n--- Creating TEST dataset ---')
test_ds = VideoQADataset(
    split='test',
    n_query=args.n_query,
    obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt,
    topK_frame=args.topK_frame,
    max_samples=None,
    verbose=True
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=PIN_MEMORY)

print('\n' + '='*60)
print('DATASET SUMMARY')
print('='*60)
print(f'Train: {len(train_ds)} samples -> {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples -> {len(val_loader)} batches')
print(f'Test:  {len(test_ds)} samples -> {len(test_loader)} batches')
print(f'pin_memory={PIN_MEMORY}')
print('='*60)

if len(train_ds) > 0:
    print('\nSanity check: Loading first batch...')
    try:
        ff, of, qns, ans, ans_id, keys = next(iter(train_loader))
        print(f'  ViT features: {ff.shape}')
        print(f'  Object features: {of.shape}')
        print(f'  Answer IDs: {ans_id[:8]}')
        print('Sanity check PASSED!')
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()

In [ ]:
# CELL 8: Model + W&B Init
print('=== CELL 8: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
scheduler = ReduceLROnPlateau(optimizer, 'max', factor=args.gamma, patience=args.patience)
model.to(device)
xe = nn.CrossEntropyLoss()
scaler = None  # keep no-AMP path for stability

save_path = os.path.join(MODEL_DIR, HF_MODEL_FILENAME)
last_ckpt_path = os.path.join(MODEL_DIR, LAST_CKPT_FILENAME)

wandb_config = {k: v for k, v in Config.__dict__.items() if not k.startswith('_') and not callable(v)}
wandb_config.update({
    'model_dir': MODEL_DIR,
    'hf_repo_id': HF_REPO_ID,
    'run_training': RUN_TRAINING,
    'max_train_samples': globals().get('MAX_TRAIN_SAMPLES', None),
    'effective_batch_size': args.bs * args.accumulation_steps,
})
if wandb.run is None:
    wandb.init(
        project=WANDB_PROJECT,
        name=f'transtr-kaggle-bs{args.bs}-acc{args.accumulation_steps}-ep{args.epoch}',
        config=wandb_config,
        save_code=True
    )
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')


In [ ]:
# CELL 9: Training + W&B + Last Checkpoint Resume
print('=== CELL 9: Training ===')

best_acc = 0.0
best_epoch = 0
start_epoch = 1
RESUME_FROM_WANDB_LAST = False

if RESUME_FROM_WANDB_LAST and wandb.run is not None:
    try:
        artifact = wandb.run.use_artifact('last-checkpoint-transtr:latest')
        art_dir = artifact.download()
        art_ckpt = os.path.join(art_dir, LAST_CKPT_FILENAME)
        if os.path.exists(art_ckpt):
            shutil.copy2(art_ckpt, last_ckpt_path)
            print(f'Pulled last checkpoint from W&B: {art_ckpt}')
    except Exception as e:
        print(f'No W&B last checkpoint artifact found: {e}')

if os.path.exists(last_ckpt_path):
    try:
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        best_acc = float(ckpt.get('best_acc', 0.0))
        best_epoch = int(ckpt.get('best_epoch', 0))
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        print(f'Resumed local last checkpoint: epoch={start_epoch-1}, best={best_acc:.2f}%')
    except Exception as e:
        print(f'Resume failed, training from scratch: {e}')

if RUN_TRAINING:
    if start_epoch > args.epoch:
        print(f'Checkpoint already reached target epoch={args.epoch}. Skipping training.')
    else:
        for ep in range(start_epoch, args.epoch + 1):
            loss, acc = train_epoch(model, optimizer, train_loader, xe, device, scaler, accumulation_steps=args.accumulation_steps)
            val_acc = eval_epoch(model, val_loader, device)
            scheduler.step(val_acc)
            print(f'Ep {ep}: Loss={loss:.4f}, Train={acc:.1f}%, Val={val_acc:.1f}%')

            ckpt = {
                'epoch': ep,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_acc': float(max(best_acc, val_acc)),
                'best_epoch': int(best_epoch),
                'config': wandb.config.as_dict() if wandb.run else {}
            }
            torch.save(ckpt, last_ckpt_path)

            if wandb.run is not None:
                wandb.log({
                    'epoch': ep,
                    'train/loss': loss,
                    'train/acc': acc,
                    'val/acc': val_acc,
                    'lr': optimizer.param_groups[0]['lr'],
                    'best_acc_so_far': max(best_acc, val_acc),
                    'accumulation_steps': args.accumulation_steps,
                    'effective_batch_size': args.bs * args.accumulation_steps,
                })

                last_art = wandb.Artifact('last-checkpoint-transtr', type='model')
                last_art.add_file(last_ckpt_path, name=LAST_CKPT_FILENAME)
                wandb.log_artifact(last_art, aliases=['latest', f'epoch-{ep}'])

            if val_acc > best_acc:
                best_acc = val_acc
                best_epoch = ep
                torch.save(model.state_dict(), save_path)
                print('  Saved best model')

                if wandb.run is not None:
                    best_art = wandb.Artifact('best-model-transtr', type='model', metadata={'best_epoch': ep, 'best_val_acc': float(val_acc)})
                    best_art.add_file(save_path, name=HF_MODEL_FILENAME)
                    wandb.log_artifact(best_art, aliases=['best', f'epoch-{ep}'])

        print(f'\nBest Val: {best_acc:.2f}% at epoch {best_epoch}')

    if wandb.run is not None:
        wandb.run.summary['best_val_acc'] = float(best_acc)
        wandb.run.summary['best_epoch'] = int(best_epoch)

    if UPLOAD_TO_HF and os.path.exists(save_path):
        try:
            api = HfApi()
            api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True)
            api.upload_file(path_or_fileobj=save_path, path_in_repo=HF_MODEL_FILENAME, repo_id=HF_REPO_ID, repo_type='model')
            print('Uploaded best model to HuggingFace')
        except Exception as e:
            print(f'Upload failed: {e}')
    elif os.path.exists(save_path):
        print(f'Best model saved locally: {save_path}')
else:
    print('Skipping training (RUN_TRAINING=False)')


In [ ]:
# CELL 10: Detailed Test Evaluation + W&B logging
print('=== CELL 10: TEST Set Evaluation ===')
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'DanielQ07/transtr-causalvid-weights'
HF_FILENAME = 'best_model.ckpt'
LOCAL_MODEL_PATH = os.path.join(MODEL_DIR, HF_FILENAME)

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"\n📥 Downloading {HF_FILENAME} from HuggingFace ({HF_REPO_ID})...")
    try:
        model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILENAME, local_dir=MODEL_DIR)
        print(f"✅ Downloaded to: {model_path}")
        LOCAL_MODEL_PATH = model_path
    except Exception as e:
        print(f"❌ Failed to download model: {e}")
else:
    print(f"📂 Found local model at: {LOCAL_MODEL_PATH}")

if os.path.exists(LOCAL_MODEL_PATH):
    print('🔧 Loading weights...')
    state = torch.load(LOCAL_MODEL_PATH, map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    msg = model.load_state_dict(state, strict=False)
    print(f'Load status: {msg}')
else:
    print('⚠️ No model weights found, evaluating random weights.')


def evaluate_detailed(model, loader, device, dataset_df):
    model.eval()
    results = {}

    print('\n📊 Running Detailed Evaluation on TEST Set...')
    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys = batch
            ff = ff.to(device)
            of = of.to(device)
            out = model(ff, of, qns, ans_word)
            preds = out.argmax(dim=-1).cpu().numpy()
            targets = ans_id.cpu().numpy()
            for key, pred, target in zip(qns_keys, preds, targets):
                results[key] = {'pred': int(pred), 'target': int(target)}

    df = dataset_df.copy()
    if 'qns_key' not in df.columns:
        df['qns_key'] = df.apply(lambda x: f"{x['video_id']}_{x['type']}", axis=1)

    # FIX: ensure answer dtype numeric before comparison
    df['answer'] = pd.to_numeric(df['answer'], errors='coerce')
    df = df.dropna(subset=['answer']).copy()
    df['answer'] = df['answer'].astype(int)

    df['pred'] = df['qns_key'].apply(lambda k: results.get(k, {}).get('pred', -1))
    missing = (df['pred'] == -1).sum()
    if missing > 0:
        print(f"⚠️ Missing predictions for {missing}/{len(df)} samples")
        df = df[df['pred'] != -1].copy()

    df['correct'] = (df['pred'] == df['answer'])
    metrics = {}

    metrics_map = {
        'Description': 'descriptive',
        'Explanation': 'explanatory',
        'Predictive-Answer': 'predictive',
        'Predictive-Reason': 'predictive_reason',
        'Counterfactual-Answer': 'counterfactual',
        'Counterfactual-Reason': 'counterfactual_reason'
    }

    print('\n' + '='*60)
    print('EVALUATION RESULTS - TEST SET')
    print('='*60)

    for name, qtype in metrics_map.items():
        sub_df = df[df['type'] == qtype]
        acc = sub_df['correct'].mean() * 100 if len(sub_df) > 0 else 0
        metrics[name] = float(acc)
        print(f"{name:<25} ==>   {acc:.2f}%  ({int(sub_df['correct'].sum())}/{len(sub_df)})")

    print('-' * 60)

    def calc_hard_metric(type_ans, type_reason, name):
        df_ans = df[df['type'] == type_ans][['video_id', 'correct']].rename(columns={'correct': 'correct_ans'})
        df_reason = df[df['type'] == type_reason][['video_id', 'correct']].rename(columns={'correct': 'correct_reason'})
        merged = pd.merge(df_ans, df_reason, on='video_id', how='inner')
        both_correct = (merged['correct_ans'] & merged['correct_reason']).sum()
        acc = both_correct / len(merged) * 100 if len(merged) > 0 else 0
        metrics[name] = float(acc)
        print(f"{name:<25} ==>   {acc:.2f}%  ({int(both_correct)}/{len(merged)} paired)")

    calc_hard_metric('predictive', 'predictive_reason', 'PAR')
    calc_hard_metric('counterfactual', 'counterfactual_reason', 'CAR')

    d_acc = metrics.get('Description', 0)
    e_acc = metrics.get('Explanation', 0)
    par_acc = metrics.get('PAR', 0)
    car_acc = metrics.get('CAR', 0)
    metrics['Acc (ALL)'] = float((d_acc + e_acc + par_acc + car_acc) / 4)

    print('-' * 60)
    print(f"{'Acc (ALL)':<25} ==>   {metrics['Acc (ALL)']:.2f}%  ((D+E+PAR+CAR)/4)")
    print('='*60)

    metric_log = {
        f"test/{k.replace(' ', '_').replace('(', '').replace(')', '')}": v
        for k, v in metrics.items()
    }
    wandb.log(metric_log)
    wandb.run.summary.update({f'test_{k}': v for k, v in metrics.items()})

    metrics_path = os.path.join(MODEL_DIR, 'test_metrics.json')
    with open(metrics_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
    final_art = wandb.Artifact('test-results-transtr', type='results')
    final_art.add_file(metrics_path, name='test_metrics.json')
    if os.path.exists(LOCAL_MODEL_PATH):
        final_art.add_file(LOCAL_MODEL_PATH, name=HF_FILENAME)
    if os.path.exists(os.path.join(MODEL_DIR, LAST_CKPT_FILENAME)):
        final_art.add_file(os.path.join(MODEL_DIR, LAST_CKPT_FILENAME), name=LAST_CKPT_FILENAME)
    wandb.log_artifact(final_art)

    plot_metrics(metrics)
    return metrics


def plot_metrics(metrics):
    keys = ['Description', 'Explanation', 'PAR', 'CAR', 'Acc (ALL)']
    values = [metrics.get(k, 0) for k in keys]
    plt.figure(figsize=(10, 6))
    bars = plt.bar(keys, values, color=sns.color_palette('viridis', len(keys)))
    plt.ylim(0, 100)
    plt.ylabel('Accuracy (%)')
    plt.title('VideoQA Performance on Test Set')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f"{bar.get_height():.1f}%", ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'test_results.png'))
    plt.show()

metrics = evaluate_detailed(model, test_loader, device, test_ds.sample_list)
print(metrics)
wandb.finish()